# 03B — Edge MFE Reference

**Goal:** verify and characterize the Edge Impulse MFE v4 frontend on the current Example Grid v2.

This notebook does **not** train a model and does **not** change 00/01/02/03A.

```text
02 Example Grid v2
      ↓
same example_uid + start_sample_16k
      ↓
15,600-sample accepted context
      ↓
Edge Impulse MFE v4
      ↓
96 × 40
      ↓
TRAIN-only diagnostics
```

The implementation is pinned locally under:

`third_party/edgeimpulse-processing-blocks/`

Expected revision:

`9b0ceb2b5d22658d3193a5b77dc773cfc8d4cab0`

This version deliberately avoids importing `mfe/dsp.py` as a Python module.  
Instead, it:

1. authenticates the pinned source files,
2. imports the vendored SpeechPy fork used by Edge Impulse,
3. explicitly reproduces the small `dsp.py` wrapper operations.

This is less fragile in Jupyter because `dsp.py` modifies `sys.path` and imports a generic package named `common`.

**Freeze target:** VERIFIED / FROZEN after the final gates; the accepted audit remains read-only.
The project baseline is **−70 dB**, selected using TRAIN-only evidence.
The pinned upstream **UI default is −52 dB**. This notebook verifies the Python
numerical implementation, not the C++ SDK, ESP32 or streaming deployment path.

## 3B.1 — Imports and project paths

In [ ]:
# Code must come from the local checkout or the clone in the remote runtime.
from pathlib import Path
import os, sys
_repo_hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
_repo = next((p for p in [_repo_hint, *_repo_hint.parents] if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Clone the Git repository in this runtime and set EDGEAI_REPO_ROOT; see 04D_experiments.ipynb")
sys.path.insert(0, str(_repo / "tools"))
from project_paths import REPO_ROOT, ProjectPaths, resolve_path, logical_relative
PATHS = ProjectPaths.from_env()
from pathlib import Path
import hashlib
import importlib
import json
import sys
import wave

import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.io import wavfile

import matplotlib.pyplot as plt

PROJECT = REPO_ROOT

sys.path.insert(0, str(PROJECT / "tools"))
from pipeline_contract import (
    authenticate_project, validate_grid, software_versions, feature_audio_contract,
    require_passed_gates, write_gated_audit, sha256_file, load_pinned_ei_wrapper,
    EXPECTED_SHA256, MANIFEST, SPLIT, GRID_AUDIT, GRID_FREEZE, EI_REVISION,
)
AUDIO_DIR = PATHS.RAW_ROOT
MANIFEST_PATH = resolve_path(PROJECT, MANIFEST)
SPLIT_PATH = resolve_path(PROJECT, SPLIT)
accepted_inputs = authenticate_project(PROJECT, audio=True, require_grid_freeze=True)
grid_generation_audit = json.loads((resolve_path(PROJECT, GRID_AUDIT)).read_text(encoding="utf-8"))

EI_ROOT = PROJECT / "third_party" / "edgeimpulse-processing-blocks"
EI_MFE_DIR = EI_ROOT / "mfe"
EI_DSP_PATH = EI_MFE_DIR / "dsp.py"
EI_PARAMS_PATH = EI_MFE_DIR / "parameters.json"
EI_SPEECHPY_DIR = EI_MFE_DIR / "third_party" / "speechpy"

required_paths = [
    AUDIO_DIR,
    MANIFEST_PATH,
    SPLIT_PATH,
    EI_DSP_PATH,
    EI_PARAMS_PATH,
    EI_SPEECHPY_DIR / "__init__.py",
    EI_SPEECHPY_DIR / "feature.py",
    EI_SPEECHPY_DIR / "processing.py",
    EI_SPEECHPY_DIR / "functions.py",
    EI_MFE_DIR / "common" / "errors.py",
]

for p in required_paths:
    assert p.exists(), f"Missing required path: {p}"

print("PROJECT:", PROJECT)
print("Manifest:", MANIFEST_PATH)
print("Edge Impulse:", EI_ROOT)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

## 3B.2 — Authenticate Example Grid v2

03B inherits the current segmentation. It must not regenerate the split or move example starts.

In [ ]:
examples = pd.read_csv(MANIFEST_PATH)
frozen_split = pd.read_csv(SPLIT_PATH)

required_manifest_cols = {
    "example_uid",
    "id",
    "name",
    "species",
    "split",
    "example_index",
    "start_sample_16k",
    "example_stride_samples",
    "reference_context_samples",
    "reference_end_sample_16k",
}

missing = required_manifest_cols - set(examples.columns)
assert not missing, f"Manifest missing columns: {sorted(missing)}"

assert len(examples) == 32645
assert examples["example_uid"].is_unique
assert not examples.duplicated(["id", "start_sample_16k"]).any()
assert (examples["example_stride_samples"] == 15360).all()
assert (examples["reference_context_samples"] == 15600).all()
assert examples.groupby("name")["split"].nunique().max() == 1

parent = (
    examples[["id", "name", "species", "split"]]
    .drop_duplicates()
    .sort_values("id")
    .reset_index(drop=True)
)

frozen_parent = (
    frozen_split[["id", "name", "species", "split"]]
    .sort_values("id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    parent,
    frozen_parent,
    check_dtype=False,
)

print("Examples:", len(examples))
print("Parent clips:", examples["id"].nunique())
print("Sources:", examples["name"].nunique())
print("Example Grid v2 authentication: PASS")
grid_gates = validate_grid(examples, frozen_split, grid_generation_audit)

## 3B.3 — Authenticate the pinned Edge Impulse source

Read the vendored parameter file and check the exact implementation details needed by this notebook.

The **UI defaults** expected for MFE v4 are:

- frame length: 20 ms
- frame stride: 10 ms
- filters: 40
- FFT: 256
- low frequency: 0 Hz
- high frequency: 0, meaning Nyquist
- noise floor: −52 dB

The project explicitly passes these geometry/frequency settings but overrides the noise floor with **−70 dB**. −52 dB is used only as an upstream baseline in diagnostics. The pinned CLI has different defaults (20 ms stride, 32 filters); it is not the project contract. The exact local source bytes are enforced by accepted SHA-256 pins, not merely fingerprinted.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

params = json.loads(EI_PARAMS_PATH.read_text(encoding="utf-8"))

assert params["info"]["latestImplementationVersion"] == 4

parameter_values = {}
for group in params["parameters"]:
    for item in group["items"]:
        parameter_values[item["param"]] = item["value"]

expected_defaults = {
    "frame_length": 0.02,
    "frame_stride": 0.01,
    "num_filters": 40,
    "fft_length": 256,
    "low_frequency": 0,
    "high_frequency": 0,
    "noise_floor_db": -52,
}

for key, expected in expected_defaults.items():
    actual = parameter_values[key]
    assert actual == expected, (
        f"{key}: expected {expected}, got {actual}"
    )

dsp_text = EI_DSP_PATH.read_text(
    encoding="utf-8",
    errors="strict",
)

required_dsp_markers = [
    "signal = (signal / 2**15).astype(np.float32)",
    "speechpy.processing.preemphasis(signal, cof=0.98, shift=1)",
    "use_old_mels = True if implementation_version <= 3 else False",
    "mfe = 10 * np.log10(mfe)",
    "mfe = (mfe - noise_floor_db) / ((-1 * noise_floor_db) + 12)",
    "mfe = np.uint8(np.around(mfe * 2**8))",
    "mfe = np.float32(mfe / 2**8)",
]

for marker in required_dsp_markers:
    assert marker in dsp_text, (
        "Pinned dsp.py no longer matches the expected v4 wrapper:\n"
        + marker
    )

source_hashes = {
    "mfe/dsp.py": sha256_file(EI_DSP_PATH),
    "mfe/parameters.json": sha256_file(EI_PARAMS_PATH),
    "mfe/third_party/speechpy/feature.py":
        sha256_file(EI_SPEECHPY_DIR / "feature.py"),
    "mfe/third_party/speechpy/processing.py":
        sha256_file(EI_SPEECHPY_DIR / "processing.py"),
    "mfe/third_party/speechpy/functions.py":
        sha256_file(EI_SPEECHPY_DIR / "functions.py"),
}

display(
    pd.DataFrame(
        [
            {"parameter": k, "value": v}
            for k, v in expected_defaults.items()
        ]
    )
)

print("\nPinned source fingerprints:")
for path, digest in source_hashes.items():
    print(path, digest)

print("\nPinned Edge Impulse source authentication: PASS")
for relative, actual in source_hashes.items():
    assert actual == EXPECTED_SHA256["third_party/edgeimpulse-processing-blocks/" + relative]
print("Accepted source hashes enforced; recorded upstream revision:", EI_REVISION)

## 3B.4 — Import only the vendored SpeechPy fork

This avoids executing the full `mfe/dsp.py` module.

In [ ]:
speechpy_parent = EI_SPEECHPY_DIR.parent

# Remove stale modules from earlier notebook attempts.
for module_name in list(sys.modules):
    if (
        module_name == "speechpy"
        or module_name.startswith("speechpy.")
        or module_name == "common"
        or module_name.startswith("common.")
    ):
        del sys.modules[module_name]

# Edge Impulse's SpeechPy fork expects mfe/ on sys.path so that
# `common.errors` resolves to mfe/common/errors.py.
for p in [
    str(EI_MFE_DIR),
    str(speechpy_parent),
]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

speechpy = importlib.import_module("speechpy")

speechpy_loaded_from = Path(
    speechpy.__file__
).resolve()

assert EI_SPEECHPY_DIR.resolve() in (
    speechpy_loaded_from,
    *speechpy_loaded_from.parents,
), (
    "Loaded the wrong speechpy package: "
    f"{speechpy_loaded_from}"
)

print("SpeechPy loaded from:", speechpy_loaded_from)
print("Vendored SpeechPy import: PASS")
for module in [speechpy.feature, speechpy.processing, speechpy.functions]:
    module_path = Path(module.__file__).resolve()
    relative = module_path.relative_to(PROJECT).as_posix()
    assert sha256_file(module_path) == EXPECTED_SHA256[relative]
pinned_generate_features = load_pinned_ei_wrapper(PROJECT, speechpy)

## 3B.5 — Shared PCM24 → 16 kHz waveform path

Keep the same project audio path already used by 02/03A:

```text
PCM24 WAV
→ SciPy left-justified int32
→ divide by 2^31
→ resample_poly(160, 441)
→ 16 kHz float waveform
```

03B does not change the original WAV decoding or resampler.

Both feature notebooks keep float64 conversion and whole-clip resampling. 02 uses
float32 only for geometry/physical validation; its waveform values are not used as a
feature cache. The filter and boundary options below explicitly preserve the prior defaults.

In [ ]:
NATIVE_SAMPLE_RATE = 44100
TARGET_SAMPLE_RATE = 16000
RESAMPLE_UP = 160
RESAMPLE_DOWN = 441

def load_core_wav_16k(clip_id):
    path = AUDIO_DIR / f"{int(clip_id)}.wav"

    if not path.exists():
        raise FileNotFoundError(path)

    with wave.open(str(path), "rb") as wf:
        channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        native_sr = wf.getframerate()
        native_frames = wf.getnframes()
        assert wf.getcomptype() == "NONE"

    if channels != 1:
        raise ValueError(
            f"{path.name}: expected mono, got {channels}"
        )
    if sample_width != 3:
        raise ValueError(
            f"{path.name}: expected PCM24, got "
            f"{sample_width} bytes/sample"
        )
    if native_sr != NATIVE_SAMPLE_RATE:
        raise ValueError(
            f"{path.name}: expected 44100 Hz, got {native_sr}"
        )

    sr, raw = wavfile.read(path)

    if sr != NATIVE_SAMPLE_RATE:
        raise ValueError(
            f"{path.name}: scipy rate mismatch {sr}"
        )
    if raw.ndim != 1:
        raise ValueError(
            f"{path.name}: expected mono array {raw.shape}"
        )
    if raw.dtype != np.int32:
        raise TypeError(
            f"{path.name}: expected PCM24 decoded as int32, "
            f"got {raw.dtype}"
        )

    assert len(raw) == native_frames and native_frames > 0
    x = raw.astype(np.float64) / float(2**31)

    y = signal.resample_poly(
        x,
        up=RESAMPLE_UP,
        down=RESAMPLE_DOWN,
        window=("kaiser", 5.0), padtype="constant", cval=0,
    )

    y = np.asarray(y, dtype=np.float64)
    assert len(y) == (native_frames * RESAMPLE_UP + RESAMPLE_DOWN - 1) // RESAMPLE_DOWN

    if not np.isfinite(y).all():
        raise ValueError(
            f"{path.name}: non-finite resampled waveform"
        )

    return y

## 3B.6 — Accepted Edge geometry

Use the same **15,600-sample context** already guaranteed by Example Grid v2.

Edge Impulse MFE v4 uses:

- 320-sample frame (20 ms)
- 160-sample stride (10 ms)
- no terminal padding

Therefore:

```text
15,360 samples → 95 frames
15,600 samples → 96 frames
```

So the accepted Edge tensor is `96 × 40`.

Note: the MFE frame itself is 320 samples, but FFT=256 causes the pinned NumPy `rfft(..., n=256)` to use only the first 256 samples of each frame.

In [ ]:
IMPLEMENTATION_VERSION = 4

CONTEXT_SAMPLES = 15600
FRAME_LENGTH_SECONDS = 0.020
FRAME_STRIDE_SECONDS = 0.010
FRAME_LENGTH_SAMPLES = 320
FRAME_STRIDE_SAMPLES = 160

NUM_FILTERS = 40
FFT_LENGTH = 256
LOW_FREQUENCY_HZ = 0
HIGH_FREQUENCY_HZ = None  # Edge parameter 0 means Nyquist
DEFAULT_NOISE_FLOOR_DB = -52

# Frozen project choice after the TRAIN-only diagnostics below.
FROZEN_NOISE_FLOOR_DB = -70

def edge_v4_frame_count(n_samples):
    x = n_samples - (
        FRAME_LENGTH_SAMPLES - FRAME_STRIDE_SAMPLES
    )
    return int(
        np.floor(
            x / FRAME_STRIDE_SAMPLES
        )
    )

assert edge_v4_frame_count(15360) == 95
assert edge_v4_frame_count(15600) == 96

print(
    "15360 samples ->",
    edge_v4_frame_count(15360),
    "MFE frames",
)
print(
    "15600 samples ->",
    edge_v4_frame_count(15600),
    "MFE frames",
)

In [ ]:
def extract_edge_context(row, waveform_16k=None):
    if waveform_16k is None:
        waveform_16k = load_core_wav_16k(
            row["id"]
        )

    start = int(row["start_sample_16k"])
    end = start + CONTEXT_SAMPLES

    if start < 0 or end > len(waveform_16k):
        raise ValueError(
            f"{row['example_uid']}: invalid Edge bounds "
            f"[{start}, {end}) for waveform length "
            f"{len(waveform_16k)}"
        )

    context = waveform_16k[start:end]

    assert len(context) == CONTEXT_SAMPLES
    assert np.isfinite(context).all()

    return context

## 3B.7 — Exact Edge Impulse MFE v4 wrapper reconstruction

This function reproduces the relevant numerical path in the pinned `mfe/dsp.py`:

```text
normalized waveform
→ float32
→ pre-emphasis 0.98, shift 1 (np.roll behavior)
→ SpeechPy MFE v4
   - rectangular frame
   - power spectrum = |rfft|² / FFT length
   - 40 Mel filters
→ 10 log10
→ noise-floor transform
→ clip [0,1]
→ round × 256
→ uint8 cast
→ clip [0,255]
→ divide by 256
```

Because our waveform is already normalized, we directly reproduce the signal **after** the wrapper's `/2**15` scaling. This avoids quantizing the project's PCM24 waveform to int16.

Pre-emphasis is applied independently per context; `np.roll` means the first
sample uses the last sample of that same context. It is not continuous-stream
pre-emphasis. Preserve this and FFT truncation as pinned Python behavior.
The direct regression below executes the authenticated upstream `generate_features`
function body with graphs disabled, avoiding only module startup side effects.

In [ ]:
def edge_mfe_v4(
    context_16k,
    noise_floor_db=FROZEN_NOISE_FLOOR_DB,
    low_frequency_hz=LOW_FREQUENCY_HZ,
    high_frequency_hz=HIGH_FREQUENCY_HZ,
    return_debug=False,
):
    x = np.asarray(
        context_16k,
        dtype=np.float64,
    )

    if x.shape != (CONTEXT_SAMPLES,):
        raise ValueError(
            f"Expected ({CONTEXT_SAMPLES},), got {x.shape}"
        )
    if not np.isfinite(x).all():
        raise ValueError("Input contains NaN/Inf")

    # Same effective dtype after dsp.py:
    # (raw / 2**15).astype(np.float32)
    x = x.astype(np.float32)

    pre = speechpy.processing.preemphasis(
        x,
        cof=0.98,
        shift=1,
    )

    mfe_linear, frame_energy, filter_freqs, filterbank = (
        speechpy.feature.mfe(
            pre,
            sampling_frequency=TARGET_SAMPLE_RATE,
            implementation_version=IMPLEMENTATION_VERSION,
            frame_length=FRAME_LENGTH_SECONDS,
            frame_stride=FRAME_STRIDE_SECONDS,
            num_filters=NUM_FILTERS,
            fft_length=FFT_LENGTH,
            low_frequency=low_frequency_hz,
            high_frequency=high_frequency_hz,
            use_old_mels=False,
        )
    )

    assert mfe_linear.shape == (96, 40)

    clipped_linear = np.clip(
        mfe_linear,
        1e-30,
        None,
    )

    mfe_db = 10 * np.log10(
        clipped_linear
    )

    normalized = (
        (mfe_db - noise_floor_db)
        / ((-1 * noise_floor_db) + 12)
    )
    normalized = np.clip(
        normalized,
        0,
        1,
    )

    # Preserve the pinned Python wrapper order exactly.
    quantized_uint8 = np.uint8(
        np.around(
            normalized * 2**8
        )
    )
    quantized_uint8 = np.clip(
        quantized_uint8,
        0,
        255,
    )

    output = np.float32(
        quantized_uint8 / 2**8
    )

    if not np.isfinite(output).all():
        raise ValueError(
            "MFE output contains NaN/Inf"
        )

    if return_debug:
        return {
            "output": output,
            "preemphasized": pre,
            "mfe_linear": mfe_linear,
            "mfe_db": mfe_db,
            "normalized_prequant": normalized,
            "quantized_uint8": quantized_uint8,
            "frame_energy": frame_energy,
            "filter_freqs": filter_freqs,
            "filterbank": filterbank,
        }

    return output

## 3B.8 — Synthetic implementation checks

These tests verify deterministic shape, finite output, frame geometry, silence behavior, and the pinned quantization path.

In [ ]:
rng = np.random.default_rng(12345)

synthetic_cases = {
    "zeros": np.zeros(
        CONTEXT_SAMPLES,
        dtype=np.float64,
    ),
    "impulse": np.r_[
        0.5,
        np.zeros(
            CONTEXT_SAMPLES - 1,
            dtype=np.float64,
        ),
    ],
    "440Hz_sine": (
        0.5
        * np.sin(
            2
            * np.pi
            * 440.0
            * np.arange(
                CONTEXT_SAMPLES,
                dtype=np.float64,
            )
            / TARGET_SAMPLE_RATE
        )
    ),
    "noise": rng.normal(
        0.0,
        0.05,
        CONTEXT_SAMPLES,
    ),
}

synthetic_summary = []

for label, x in synthetic_cases.items():
    debug = edge_mfe_v4(
        x,
        return_debug=True,
    )

    feat = debug["output"]

    assert feat.shape == (96, 40)
    assert np.isfinite(feat).all()
    assert feat.dtype == np.float32
    assert feat.min() >= 0
    assert feat.max() <= (255 / 256)

    synthetic_summary.append({
        "input": label,
        "shape": str(feat.shape),
        "min": float(feat.min()),
        "max": float(feat.max()),
        "zero_fraction": float(
            np.mean(feat == 0)
        ),
    })

synthetic_summary = pd.DataFrame(
    synthetic_summary
)

display(synthetic_summary)

print("Synthetic Edge MFE checks: PASS")

# Direct regression against the original, authenticated dsp.py function body.
# Scale by 2**15 without int16 quantization: the wrapper restores the same input.
wrapper_parity_results = []
def check_wrapper_parity(label, context, floor):
    reference = pinned_generate_features(
        4, False, np.asarray(context, dtype=np.float64) * 2**15, ["audio"],
        TARGET_SAMPLE_RATE, FRAME_LENGTH_SECONDS, FRAME_STRIDE_SECONDS,
        NUM_FILTERS, FFT_LENGTH, LOW_FREQUENCY_HZ, 0, 101, floor)
    expected = np.asarray(reference["features"], dtype=np.float32).reshape(96, 40)
    actual = edge_mfe_v4(context, noise_floor_db=floor)
    assert np.array_equal(actual, expected), f"Pinned wrapper regression: {label}, {floor}"
    wrapper_parity_results.append({"case":label, "noise_floor_db":floor,
                                   "max_abs_diff":float(np.max(np.abs(actual - expected)))})
for floor in [DEFAULT_NOISE_FLOOR_DB, -65, FROZEN_NOISE_FLOOR_DB]:
    for label, context in synthetic_cases.items():
        check_wrapper_parity(label, context, floor)
    for index in [255, 319, 15200, CONTEXT_SAMPLES - 1]:
        context = np.zeros(CONTEXT_SAMPLES, dtype=np.float64)
        context[index] = 0.5
        check_wrapper_parity(f"boundary_impulse_{index}", context, floor)
print("Direct pinned Python wrapper synthetic/boundary regression: PASS")

### Explicit quantization-boundary check

The pinned Python wrapper performs `uint8` conversion before clipping.  
Therefore a normalized value of exactly `1.0` rounds to `256` and wraps to `0`.

This is an implementation behavior check, **not** a statement that real mosquito features necessarily hit this boundary.

In [ ]:
boundary = np.array(
    [
        0.0,
        0.5,
        255.4 / 256.0,
        255.5 / 256.0,
        1.0,
    ],
    dtype=np.float64,
)

pinned_q = np.uint8(
    np.around(
        boundary * 2**8
    )
)
pinned_q = np.clip(
    pinned_q,
    0,
    255,
)
pinned_dequant = np.float32(
    pinned_q / 2**8
)

boundary_table = pd.DataFrame({
    "normalized_prequant": boundary,
    "uint8_after_round_cast": pinned_q,
    "dequantized": pinned_dequant,
})

display(boundary_table)

assert pinned_q[-1] == 0
print("Pinned quantization-boundary behavior: PASS")

## 3B.9 — Real Core examples

Check one deterministic TRAIN example per species.

In [ ]:
train_examples = examples[
    examples["split"] == "train"
].copy()

representatives = (
    train_examples
    .sort_values(
        [
            "species",
            "id",
            "start_sample_16k",
        ]
    )
    .groupby(
        "species",
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

clip_cache = {}
real_summary = []

for _, row in representatives.iterrows():
    clip_id = int(row["id"])

    if clip_id not in clip_cache:
        clip_cache[clip_id] = (
            load_core_wav_16k(
                clip_id
            )
        )

    context = extract_edge_context(
        row,
        clip_cache[clip_id],
    )

    feat = edge_mfe_v4(
        context
    )

    assert feat.shape == (96, 40)
    assert np.isfinite(feat).all()

    real_summary.append({
        "example_uid":
            row["example_uid"],
        "species":
            row["species"],
        "shape":
            str(feat.shape),
        "min":
            float(feat.min()),
        "max":
            float(feat.max()),
        "mean":
            float(feat.mean()),
        "std":
            float(feat.std()),
        "zero_fraction":
            float(
                np.mean(feat == 0)
            ),
    })

real_summary = pd.DataFrame(
    real_summary
)

display(real_summary)

print("Real Core MFE checks: PASS")
for _, row in representatives.iterrows():
    context = extract_edge_context(row, clip_cache[int(row["id"])])
    check_wrapper_parity(row["example_uid"], context, FROZEN_NOISE_FLOOR_DB)
print("Direct pinned wrapper real TRAIN regression: PASS")

## 3B.10 — Visual inspection of one real example

In [ ]:
row = representatives.iloc[0]

waveform_16k = load_core_wav_16k(
    row["id"]
)

context = extract_edge_context(
    row,
    waveform_16k,
)

debug = edge_mfe_v4(
    context,
    return_debug=True,
)

feat = debug["output"]

print("Example:", row["example_uid"])
print("Species:", row["species"])
print("Context shape:", context.shape)
print("MFE shape:", feat.shape)

plt.figure(figsize=(10, 5))
plt.imshow(
    feat.T,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
)
plt.xlabel("Time frame")
plt.ylabel("Mel filter")
plt.title(
    "Edge Impulse MFE v4 — frozen project noise floor −70 dB"
)
plt.colorbar(
    label="quantized/dequantized feature"
)
plt.tight_layout()
plt.show()

## 3B.11 — TRAIN-only noise-floor sensitivity

Do **not** automatically accept Edge Impulse's default noise floor of `−52 dB`.

This diagnostic performs an initial bounded sensitivity analysis using a deterministic, approximately balanced subset of TRAIN examples only.

The following candidate noise floors are compared:

`−52, −55, −60, −65, −70 dB`

For each candidate, measure:

- the fraction of final MFE bins equal to zero,
- the number of examples whose final MFE tensor is entirely zero,
- the fraction of pre-quantized values close enough to the upper boundary to round to 256 before the pinned `uint8` cast,
- the mean and standard deviation of the final MFE features.

This stage is used only to identify plausible noise-floor candidates. It does **not** freeze the final value.

No validation or test examples are used, no examples are removed, and no classifier is trained.

The remaining candidates are subsequently evaluated on the complete TRAIN example set before the 03B frontend contract is frozen.

In [ ]:
AUDIT_EXAMPLES_PER_SPECIES = 32

audit_parts = []

for species, group in (
    train_examples
    .sort_values(
        [
            "species",
            "id",
            "start_sample_16k",
        ]
    )
    .groupby(
        "species",
        sort=True,
    )
):
    n = min(
        AUDIT_EXAMPLES_PER_SPECIES,
        len(group),
    )

    positions = np.linspace(
        0,
        len(group) - 1,
        n,
        dtype=int,
    )

    audit_parts.append(
        group.iloc[positions]
    )

audit_examples = (
    pd.concat(
        audit_parts,
        ignore_index=True,
    )
    .drop_duplicates("example_uid")
    .reset_index(drop=True)
)

for clip_id in (
    audit_examples["id"]
    .astype(int)
    .unique()
):
    if clip_id not in clip_cache:
        clip_cache[clip_id] = (
            load_core_wav_16k(
                clip_id
            )
        )

print(
    "TRAIN examples in diagnostic:",
    len(audit_examples),
)
display(
    audit_examples[
        "species"
    ].value_counts()
)

In [ ]:
NOISE_FLOOR_CANDIDATES_DB = [
    -52,
    -55,
    -60,
    -65,
    -70,
]

ROUND_TO_256_THRESHOLD = (
    255.5 / 256.0
)

sweep_rows = []

for noise_floor_db in (
    NOISE_FLOOR_CANDIDATES_DB
):
    all_final = []
    all_zero_count = 0
    near_one_fractions = []

    for _, row in (
        audit_examples.iterrows()
    ):
        context = extract_edge_context(
            row,
            clip_cache[
                int(row["id"])
            ],
        )

        debug = edge_mfe_v4(
            context,
            noise_floor_db=
                noise_floor_db,
            return_debug=True,
        )

        final = debug["output"]
        preq = debug[
            "normalized_prequant"
        ]

        all_final.append(
            final.ravel()
        )

        all_zero_count += int(
            np.all(final == 0)
        )

        near_one_fractions.append(
            float(
                np.mean(
                    preq
                    >= ROUND_TO_256_THRESHOLD
                )
            )
        )

    values = np.concatenate(
        all_final
    )

    sweep_rows.append({
        "noise_floor_db":
            noise_floor_db,
        "zero_fraction":
            float(
                np.mean(values == 0)
            ),
        "all_zero_examples":
            int(all_zero_count),
        "near_one_fraction":
            float(
                np.mean(
                    near_one_fractions
                )
            ),
        "mean":
            float(values.mean()),
        "std":
            float(values.std()),
        "min":
            float(values.min()),
        "max":
            float(values.max()),
    })

noise_floor_sweep = pd.DataFrame(
    sweep_rows
)

display(noise_floor_sweep)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    noise_floor_sweep[
        "noise_floor_db"
    ],
    noise_floor_sweep[
        "zero_fraction"
    ],
    marker="o",
)
plt.xlabel("Noise floor (dB)")
plt.ylabel(
    "Fraction of final MFE bins = 0"
)
plt.title(
    "TRAIN-only MFE sparsity "
    "vs noise floor"
)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(
    noise_floor_sweep[
        "noise_floor_db"
    ],
    noise_floor_sweep[
        "all_zero_examples"
    ],
    marker="o",
)
plt.xlabel("Noise floor (dB)")
plt.ylabel(
    "All-zero examples"
)
plt.title(
    "TRAIN-only all-zero MFE "
    "examples"
)
plt.tight_layout()
plt.show()

## 3B.12 — Frequency-range status

Keep the pinned Edge Impulse frequency-range defaults for the frozen project baseline:

```text
low = 0 Hz
high = Nyquist = 8,000 Hz
```

Do not introduce the historical 100/150/200/300 Hz low-cut trials yet.

The −70 dB noise floor has been selected under **current Example Grid v2**.
Frequency-range changes would define a separate experiment requiring explicit approval.

## 3B.13 — Full-TRAIN confirmation of the candidate noise floor

The previous diagnostic used a balanced subset of TRAIN examples to identify plausible Edge MFE noise-floor settings.

Before freezing the 03B frontend contract, evaluate the two remaining candidates, **−65 dB** and **−70 dB**, over the complete TRAIN example set.

This step is restricted to the **TRAIN split only** and does not use validation or test data for frontend selection.

For each candidate, measure:

- the fraction of final MFE bins equal to zero,
- the number of examples whose final MFE tensor is entirely zero,
- the fraction of pre-quantized values close enough to the upper boundary to round to 256 before the pinned `uint8` cast.

No examples are removed, no classifier is trained, and no parameters are selected automatically in this cell.

The purpose is only to confirm that the candidate noise floor remains numerically well behaved at full TRAIN scale before the 03B contract is frozen.

In [ ]:
FULL_TRAIN_FLOORS = [-65, -70]
assert len(train_examples) == 22613 and train_examples["split"].eq("train").all()

# Release diagnostic waveforms, then retain only one decoded/resampled clip.
clip_cache.clear()
accumulators = {floor: {"total_bins":0, "zero_bins":0, "all_zero_examples":0,
                        "near_one_bins":0, "examples":0} for floor in FULL_TRAIN_FLOORS}
for clip_id, clip_examples in train_examples.groupby("id", sort=False):
    waveform = load_core_wav_16k(clip_id)
    for _, row in clip_examples.iterrows():
        context = extract_edge_context(row, waveform)
        for floor in FULL_TRAIN_FLOORS:
            debug = edge_mfe_v4(context, noise_floor_db=floor, return_debug=True)
            final = debug["output"]
            preq = debug["normalized_prequant"]
            counts = accumulators[floor]
            counts["examples"] += 1
            counts["total_bins"] += int(final.size)
            counts["zero_bins"] += int(np.count_nonzero(final == 0))
            counts["all_zero_examples"] += int(np.all(final == 0))
            counts["near_one_bins"] += int(np.count_nonzero(preq >= ROUND_TO_256_THRESHOLD))
    del context, waveform

full_train_rows = []
for floor in FULL_TRAIN_FLOORS:
    counts = accumulators[floor]
    assert counts["examples"] == 22613
    assert counts["total_bins"] == 22613 * 96 * 40
    full_train_rows.append({"noise_floor_db":floor, "examples":counts["examples"],
        "zero_fraction":counts["zero_bins"] / counts["total_bins"],
        "all_zero_examples":counts["all_zero_examples"],
        "near_one_fraction":counts["near_one_bins"] / counts["total_bins"],
        "zero_bins":counts["zero_bins"], "near_one_bins":counts["near_one_bins"],
        "total_bins":counts["total_bins"]})
full_train_check = pd.DataFrame(full_train_rows)
# Accepted pre-review evidence: exact counts and fractions displayed to six decimals.
for floor, all_zero, displayed_zero_fraction in [(-65, 11, 0.870198), (-70, 0, 0.713330)]:
    result = full_train_check.set_index("noise_floor_db").loc[floor]
    assert int(result["all_zero_examples"]) == all_zero, "Approved TRAIN evidence changed; STOP"
    assert result["near_one_fraction"] == 0.0, "Approved boundary evidence changed; STOP"
    assert round(float(result["zero_fraction"]), 6) == displayed_zero_fraction, "Approved TRAIN sparsity changed; STOP"
display(full_train_check)

## 3B.14 — Freeze the Edge MFE frontend contract

The full-TRAIN diagnostic confirms the final Edge MFE noise-floor choice.

At **−65 dB**, 11 of 22,613 TRAIN examples still produced entirely zero-valued MFE tensors. At **−70 dB**, no TRAIN example was entirely zero, while the upper-bound diagnostic remained inactive (`near_one_fraction = 0`).

Therefore, **−70 dB is selected as the frozen baseline noise floor** for the Edge MFE branch.

This value is not claimed to be globally optimal. It is the selected baseline from the predefined TRAIN-only diagnostic range, chosen to avoid complete feature collapse while preserving the pinned Edge Impulse MFE v4 processing behavior.

The frozen 03B frontend contract is:

- sample rate: 16 kHz
- context: 15,600 samples
- frame length: 320 samples (20 ms)
- frame stride: 160 samples (10 ms)
- FFT length: 256
- Mel filters: 40
- frequency range: 0–8,000 Hz
- noise floor: −70 dB
- output tensor: 96 × 40

No validation or test examples were used to select this configuration.

This is a comparison of complete frontend configurations. MFE's pre-emphasis,
rectangular framing/FFT truncation, power spectrum, 40 filters, range and feature
quantization intentionally differ from 03A. Equal IDs/starts do not imply equal
spectral support or an experiment isolating a single mathematical operation.

## 3B.15 — Final validation and freeze gate

Every required check, including full-TRAIN confirmation and source reauthentication,
must pass while the authoritative JSON is retained unchanged. −70 dB is the approved project
baseline; −52 dB remains the pinned upstream UI default.

In [ ]:
assert len(examples) == 32645
assert len(train_examples) == 22613
assert edge_v4_frame_count(CONTEXT_SAMPLES) == 96

assert FROZEN_NOISE_FLOOR_DB == -70
assert synthetic_summary["shape"].eq("(96, 40)").all()
assert real_summary["shape"].eq("(96, 40)").all()

selected = full_train_check[
    full_train_check["noise_floor_db"] == FROZEN_NOISE_FLOOR_DB
].iloc[0]

assert int(selected["examples"]) == 22613
assert int(selected["all_zero_examples"]) == 0
assert float(selected["near_one_fraction"]) == 0.0

assert DEFAULT_NOISE_FLOOR_DB == -52
assert len(wrapper_parity_results) == 28
assert all(r["max_abs_diff"] == 0.0 for r in wrapper_parity_results)
assert len(audit_examples) == 128 and audit_examples["split"].eq("train").all()
assert representatives["split"].eq("train").all()
assert edge_mfe_v4.__defaults__[0] == FROZEN_NOISE_FLOOR_DB
accepted_inputs = authenticate_project(PROJECT, audio=True, require_grid_freeze=True)
final_gates = dict(grid_gates, accepted_input_and_source_hashes=True, all_wav_identities=True,
    synthetic_checks=True, pinned_wrapper_regression=True, real_train_checks=True,
    quantization_boundary=True, full_train_confirmation=True,
    selected_minus70_no_collapse_or_wrap=True, train_only_selection=True)
require_passed_gates(final_gates)
print("03B final validation gates: PASS; audit eligible for persistence")

## 3B.16 — Persist the successfully validated freeze audit

Record the project contract, authenticated inputs/source bytes, full-TRAIN −65/−70
confirmation, selection rationale, direct upstream regression and final gate results.
Any change to accepted pre-review numerical summaries stops before replacing the audit.

In [ ]:
AUDIT_PATH = resolve_path(PROJECT, "data/audits/current/03B_edge_mfe_implementation_audit.json")
audit_report = {
    "status":"VERIFIED_FROZEN", "model_training_performed":False,
    "authenticated_inputs":accepted_inputs, "final_gates":final_gates,
    "software":software_versions(), "audio_preprocessing":feature_audio_contract(),
    "example_grid":{"manifest":MANIFEST, "manifest_sha256":sha256_file(MANIFEST_PATH),
                    "examples":int(len(examples)), "stride_samples":15360, "context_samples":CONTEXT_SAMPLES},
    "frozen_split":{"path":SPLIT, "sha256":sha256_file(SPLIT_PATH)},
    "edge_impulse":{"expected_revision":EI_REVISION, "authentication":"accepted SHA-256 source pins enforced",
                    "implementation_version":IMPLEMENTATION_VERSION, "source_hashes_sha256":source_hashes,
                    "ui_defaults":expected_defaults, "cli_defaults_used":False,
                    "parity_scope":"Pinned Python numerical function body; not C++/ESP32/streaming parity"},
    "frozen_contract":{"sample_rate_hz":TARGET_SAMPLE_RATE, "context_samples":CONTEXT_SAMPLES,
        "frame_length_samples":FRAME_LENGTH_SAMPLES, "frame_stride_samples":FRAME_STRIDE_SAMPLES,
        "frames":edge_v4_frame_count(CONTEXT_SAMPLES), "num_filters":NUM_FILTERS, "fft_length":FFT_LENGTH,
        "low_frequency_hz":LOW_FREQUENCY_HZ,
        "high_frequency_hz":TARGET_SAMPLE_RATE // 2 if HIGH_FREQUENCY_HZ is None else HIGH_FREQUENCY_HZ,
        "noise_floor_db":FROZEN_NOISE_FLOOR_DB,
        "edge_impulse_default_noise_floor_db":DEFAULT_NOISE_FLOOR_DB,
        "output_shape":[96,NUM_FILTERS], "output_dtype":"float32",
        "preemphasis":{"coefficient":0.98,"shift":1,"scope":"per_context","boundary":"np.roll"},
        "wrapper_input_dtype":"float32", "window":"rectangular",
        "fft_truncates_frame_to_samples":FFT_LENGTH, "spectrum":"abs(rfft)**2 / FFT_LENGTH",
        "mel_implementation":"vendored SpeechPy v4; use_old_mels=False",
        "zero_handling":"vendored zero_handling, followed by clip at 1e-30",
        "normalization":"clip((10*log10(mfe)-floor)/(-floor+12),0,1)",
        "quantization":"around(x*256), uint8 cast, clip 0..255, float32 /256; wrap preserved"},
    "synthetic_checks":synthetic_summary.to_dict("records"),
    "pinned_wrapper_parity":wrapper_parity_results,
    "real_train_examples":real_summary.to_dict("records"),
    "train_only_noise_floor_sweep":noise_floor_sweep.to_dict("records"),
    "diagnostic_subset":{"split":"train", "example_uids":audit_examples["example_uid"].tolist(),
                         "selection":"32 evenly spaced sorted TRAIN examples per species"},
    "full_train_noise_floor_confirmation":full_train_check.to_dict("records"),
    "noise_floor_selection":{"selected_db":FROZEN_NOISE_FLOOR_DB, "upstream_ui_default_db":DEFAULT_NOISE_FLOOR_DB,
        "candidate_range_db":NOISE_FLOOR_CANDIDATES_DB, "confirmation_candidates_db":FULL_TRAIN_FLOORS,
        "rationale":"-65 dB leaves 11/22613 all-zero TRAIN tensors; -70 dB leaves zero, with zero near-one bins. Bounded TRAIN-only anti-collapse baseline, not globally optimal.",
        "validation_or_test_used":False, "examples_removed":0},
}
# Verify accepted numerical evidence without replacing the fixed audit.
prior_audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
if prior_audit is not None:
    for key in ["synthetic_checks", "real_train_examples", "train_only_noise_floor_sweep"]:
        assert audit_report[key] == prior_audit[key], f"Accepted numerical evidence changed: {key}; STOP"
    if "full_train_noise_floor_confirmation" in prior_audit:
        assert audit_report["full_train_noise_floor_confirmation"] == prior_audit["full_train_noise_floor_confirmation"]
print("Gates passed; accepted audit retained unchanged (read-only workflow).")
print("03B_EDGE_MFE: VERIFIED / FROZEN — project −70 dB; upstream UI default −52 dB")